In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
[results_1l, results_2l, results_3l],
ignore_index=True
)
#results = results_3l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,R2diff_ZZx1_theta,R2_ZZx2_theta,R2diff_ZZx2_theta,...,R2_LSG_1_theta,R2diff_LSG_1_theta,R2_LSG_2_theta,R2diff_LSG_2_theta,R2_ZZx1_inv_theta,R2diff_ZZx1_inv_theta,R2_zzx2_inv2_theta,R2diff_zzx2_inv2_theta,R2_semiCirc_theta,R2diff_semiCirc_theta
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed9468,[1],0.5,0.5,0.01,9468,0.812644,0.637816,-0.866878,0.383926,...,0.779333,0.583208,-6.126531,0.283605,-0.074995,0.286269,-7.858811,-0.052684,-36.864434,-0.346101
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed9583,[1],0.5,0.5,0.01,9583,0.335420,0.663848,-0.481250,0.430286,...,0.804239,0.598235,-3.419127,0.337881,-0.559936,0.406081,-8.470865,0.039391,-29.539623,-0.180141
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed2302,[1],0.5,0.5,0.01,2302,0.606371,0.635331,-0.861561,0.384327,...,0.851640,0.587906,-4.496989,0.313197,-0.196650,0.326580,-7.976547,-0.028246,-33.820807,-0.277783
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed134,[1],0.5,0.5,0.01,134,0.723993,0.643883,-0.936011,0.385927,...,0.724940,0.597476,-5.251368,0.309798,-0.112615,0.314128,-7.958574,-0.062566,-38.820195,-0.377207
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed9940,[1],0.5,0.5,0.01,9940,0.167741,0.617201,-0.056641,0.409887,...,-0.444151,0.526502,-3.264614,0.288599,-0.775709,0.352238,-7.955050,0.112877,-17.804839,0.011120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6474,model_arch51-20-13_r0.9_Ld0.7_Lp0.3_seed9240,"[51, 20, 13]",0.7,0.3,0.90,9240,0.884278,0.719823,0.950213,0.468540,...,0.757303,0.573357,-3.555860,0.311616,-2.625067,0.383831,-16.407439,0.060114,-33.764539,-0.284818
6475,model_arch51-20-13_r0.9_Ld0.7_Lp0.3_seed7611,"[51, 20, 13]",0.7,0.3,0.90,7611,0.861331,0.796338,0.758808,0.454275,...,0.792078,0.613375,-3.243134,0.286141,-0.304951,0.444418,-13.350959,0.055636,-39.258631,-0.394718
6476,model_arch51-20-13_r0.9_Ld0.7_Lp0.3_seed1425,"[51, 20, 13]",0.7,0.3,0.90,1425,0.741843,0.790561,0.567459,0.472901,...,0.565134,0.578976,-1.727293,0.350847,-0.001188,0.456233,-13.371409,0.034962,-44.052132,-0.488062
6477,model_arch51-20-13_r0.9_Ld0.7_Lp0.3_seed7433,"[51, 20, 13]",0.7,0.3,0.90,7433,0.864631,0.771576,0.608293,0.466972,...,0.864604,0.630972,-2.650010,0.400171,-3.178403,0.495817,-19.407550,0.021209,-32.722528,-0.222885


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
#    "ZZy1":     "Test",
#    "ZZy2":     "Test",
#     "LSG-1":    "Test",
#     "LSG-2":    "Test",
#     "ZZx1-inv": "Test",
#     "ZZx2-inv2": "Test",
#    "semiCirc": "Test",
}
def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.3
w_train = 0.3
w_test = 0.3

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        #- 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
3928,model_arch70-40_r0.01_Ld0.7_Lp0.3_seed3975,"[70, 40]",0.938095,0.976807,0.955218,0.861036
3900,model_arch70-39_r0.01_Ld0.7_Lp0.3_seed3206,"[70, 39]",0.927739,0.970275,0.950227,0.854473
3626,model_arch70-30_r0.01_Ld0.7_Lp0.3_seed8743,"[70, 30]",0.941110,0.955836,0.944624,0.852471
3909,model_arch70-40_r0.01_Ld0.5_Lp0.5_seed5920,"[70, 40]",0.922740,0.972431,0.942579,0.851325
4645,model_arch70-45_r0.9_Ld0.7_Lp0.3_seed7948,"[70, 45]",0.927427,0.948980,0.957328,0.850120



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZxReto_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
3928,model_arch70-40_r0.01_Ld0.7_Lp0.3_seed3975,"[70, 40]",0.938095,0.976807,0.955218,0.938095,0.976807,0.955218,0.861036
3900,model_arch70-39_r0.01_Ld0.7_Lp0.3_seed3206,"[70, 39]",0.927739,0.970275,0.950227,0.927739,0.970275,0.950227,0.854473
3626,model_arch70-30_r0.01_Ld0.7_Lp0.3_seed8743,"[70, 30]",0.941110,0.955836,0.944624,0.941110,0.955836,0.944624,0.852471
3909,model_arch70-40_r0.01_Ld0.5_Lp0.5_seed5920,"[70, 40]",0.922740,0.972431,0.942579,0.922740,0.972431,0.942579,0.851325
4645,model_arch70-45_r0.9_Ld0.7_Lp0.3_seed7948,"[70, 45]",0.927427,0.948980,0.957328,0.927427,0.948980,0.957328,0.850120


In [5]:
final_table.to_excel("BestModels-3tst.xlsx")

In [6]:
# ============================================
# MÉDIA, DESVIO, MÍNIMO E MÁXIMO
# ============================================
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
   "ZZy1":     "Test",
   "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv2": "Test",
   "semiCirc": "Test",
}
summary_tables = {}

for target in TARGETS:

    top_df = results.copy()
    rows = []

    for s in SETS_CATEGORY.keys():

        r2_col = f"R2_{s.replace('-', '_')}_{target}"
        mse_col = f"MSE_{s.replace('-', '_')}_{target}"

        row = {
            "Set": s,
            "Category": SETS_CATEGORY[s]
        }

        # =========================
        # R²
        # =========================
        if r2_col in top_df.columns:
            row["R2_mean"] = top_df[r2_col].mean()
            row["R2_std"]  = top_df[r2_col].std()
            row["R2_min"]  = top_df[r2_col].min()
            row["R2_max"]  = top_df[r2_col].max()
        else:
            row["R2_mean"] = np.nan
            row["R2_std"]  = np.nan
            row["R2_min"]  = np.nan
            row["R2_max"]  = np.nan

        # =========================
        # MSE
        # =========================
        if mse_col in top_df.columns:
            row["MSE_mean"] = top_df[mse_col].mean()
            row["MSE_std"]  = top_df[mse_col].std()
            row["MSE_min"]  = top_df[mse_col].min()
            row["MSE_max"]  = top_df[mse_col].max()
        else:
            row["MSE_mean"] = np.nan
            row["MSE_std"]  = np.nan
            row["MSE_min"]  = np.nan
            row["MSE_max"]  = np.nan

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    summary_tables[target] = summary_df

    # =========================
    # MOSTRA SOMENTE A TABELA
    # =========================
    display(
        summary_df.style.format({
            "R2_mean": "{:.4f}",
            "R2_std":  "{:.4f}",
            "R2_min":  "{:.4f}",
            "R2_max":  "{:.4f}",

            "MSE_mean": "{:.6f}",
            "MSE_std":  "{:.6f}",
            "MSE_min":  "{:.6f}",
            "MSE_max":  "{:.6f}"
        })
    )

AttributeError: The '.style' accessor requires jinja2